# jG_01 - Federated SFM (tingkat) vs metode FL yang sudah ada (JURNAL)
**Tujuan:** menguji apakah pemetaan bertingkat dengan agregasi sadar-tingkat (FSFM) lebih baik daripada irisan 9 fitur,
isi-nol, dan lapisan pertama lokal. Juga membekukan angka inti dengan **log1p + Adam** (hasil jF_01: z-score + SGD kurang terlatih).
Rujukan desain: `federated/fsfm-design.md`. Tiga klien: `cic2017`, `cic2018` (76 fitur), `unsw` (9 fitur).

| Kode | Metode | Keterangan |
|---|---|---|
| M0 | local-only | tiap klien sendiri |
| M1 | centralized-pad | acuan batas atas, melanggar privasi |
| M2 | FedAvg-int9 | irisan 9 fitur |
| M3 | FedAvg-pad76 | semua fitur, klien tanpa ekstensi diisi nol, rata-rata biasa |
| M4 | FedPer-ownlayer | lapisan pertama lokal |
| M5 | FedAvg-FSFMtier | **usulan**: pad76 + rata-rata baris ekstensi hanya atas pemiliknya |
| M6-M8 | FedProx (mu 0,01) untuk M2, M3, M5 | |

**Kriteria dibaca (disepakati SEBELUM melihat angka)**, MCC rata-rata tiga klien per seed, 5 seed, CI 95% t-paired:
- **H1:** M5 - M2 >= 0,05 dan batas bawah CI > 0, dan MCC klien unsw pada M5 tidak turun > 0,02 dari M2.
- **H2:** M5 - M3 >= 0,02 dan M5 - M4 >= 0,02, kedua batas bawah CI > 0.
- H1 gagal: ide pemetaan bertingkat tidak layak untuk Q1; laporkan apa adanya.

**Prasyarat:** data near (`jA_01`) dan far (`s01`) sudah di S3. **Output:** `journal_out_fsfm/*.csv` dan S3 `federated_journal/fsfm/results/`.
Uji kering lokal: `FSFM_MOCK=1 FSFM_FAST=1`. Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess, os
MOCK = os.environ.get('FSFM_MOCK', '0') == '1'   # uji kering dengan data sintetik (tanpa S3)
FAST = os.environ.get('FSFM_FAST', '0') == '1'   # konfigurasi kecil untuk uji cepat
need = [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('scipy','scipy')] + ([] if MOCK else [('boto3','boto3')])
for mod, pkg in need:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)
import io, json, time, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ---------------- KONFIG ----------------
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
FAR_ROOT='unsw-far/federated_survey'            # klien: cic, unsw (9 fitur selaras)
NEAR_ROOT='federated_journal/neardomain'        # klien: cic2017, cic2018 (76 fitur)
OUT='federated_journal/fsfm'
OUTDIR='journal_out_fsfm'; os.makedirs(OUTDIR, exist_ok=True)
CLIENTS=['cic2017','cic2018','unsw']
SEEDS=[42,13,101,202,303]
ROUNDS=30; LOCAL_EPOCHS=1; LR=1e-3; BATCH=256; HID=[128,64]; MU=0.01
MAX_TRAIN=100000; LOCAL_ONLY_EPOCHS=15; CENTRAL_EPOCHS=10
if FAST:
    SEEDS=[42,13]; ROUNDS=3; MAX_TRAIN=3000; LOCAL_ONLY_EPOCHS=2; CENTRAL_EPOCHS=2
print('MOCK =',MOCK,'| FAST =',FAST,'| seeds =',SEEDS,'| rounds =',ROUNDS)


In [ ]:
NINE=['flow duration','total fwd packets','total backward packets',
      'total length of fwd packets','total length of bwd packets',
      'fwd packet length mean','bwd packet length mean','flow bytes/s','bwd packets/s']
ALIGN_KEYS=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
D_CORE=9
if not MOCK:
    import boto3
    s3=boto3.client('s3',region_name=REGION)
    def npz(key): return np.load(io.BytesIO(s3.get_object(Bucket=BUCKET,Key=key)['Body'].read()),allow_pickle=True)
    def load_raw(root,c):
        D={}
        for sp in ('train','test'):
            d=npz(f'{root}/data/{c}/{c}_{sp}.npz'); D[sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int)); D['feats']=[str(f) for f in d['feats']]
        return D

def mock_client(n,d,seed,shift):
    rng=np.random.RandomState(seed); Z=rng.randn(n,d)
    y=(Z[:,:4].sum(1)+0.7*rng.randn(n)>0.5).astype(int)
    Z[y==1,:6]+=0.8
    if d>15: Z[y==1,9:15]+=0.8
    return np.expm1(np.abs(Z+shift)*1.5), y

DATA={}
rs0=np.random.RandomState(0)
def subsample(X,y):
    if MAX_TRAIN and len(y)>MAX_TRAIN:
        i=rs0.choice(len(y),MAX_TRAIN,replace=False); return X[i],y[i]
    return X,y
if MOCK:
    names76=NINE+[f'ext_{i}' for i in range(67)]
    for k,c in enumerate(('cic2017','cic2018')):
        Xtr,ytr=mock_client(4000,76,10+k,0.1*k); Xte,yte=mock_client(1500,76,20+k,0.1*k)
        DATA[c]=dict(train=(Xtr,ytr),test=(Xte,yte),feats=names76)
    Xtr,ytr=mock_client(4000,9,30,0.3); Xte,yte=mock_client(1500,9,31,0.3)
    DATA['unsw']=dict(train=(Xtr,ytr),test=(Xte,yte),feats=NINE)
else:
    for c in ('cic2017','cic2018'):
        D=load_raw(NEAR_ROOT,c); f=D['feats']
        core=[f.index(n) for n in NINE]; ext=[i for i in range(len(f)) if i not in core]; order=core+ext
        DATA[c]=dict(train=subsample(D['train'][0][:,order],D['train'][1]),
                     test=(D['test'][0][:,order],D['test'][1]),feats=[f[i] for i in order])
    U=load_raw(FAR_ROOT,'unsw')
    assert U['feats']==ALIGN_KEYS, f'urutan fitur UNSW tidak sesuai ALIGN: {U["feats"]}'
    DATA['unsw']=dict(train=subsample(*U['train']),test=U['test'],feats=NINE)
D_EXT=len(DATA['cic2017']['feats'])-D_CORE
assert len(DATA['cic2018']['feats'])-D_CORE==D_EXT and DATA['cic2017']['feats']==DATA['cic2018']['feats']
for c in CLIENTS:
    X,y=DATA[c]['train']; print(c,'train',X.shape,'pos=%.3f'%y.mean(),'| test',DATA[c]['test'][0].shape,'| max kolom 0..2 =',np.round(X[:,:3].max(0),1))
print('D_CORE =',D_CORE,'| D_EXT =',D_EXT)


In [ ]:
# ---------------- mesin MLP numpy (logika sama dengan jF_01) ----------------
def prep(Xtr,Xte):
    f=lambda X: np.sign(X)*np.log1p(np.abs(X))
    A=f(Xtr); B=f(Xte); mu=A.mean(0); sd=A.std(0); sd[sd==0]=1.0
    return (A-mu)/sd,(B-mu)/sd
def init_w(layers,seed):
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(layers[:-1],layers[1:]): W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def fwd_all(W,X):
    acts=[X]; h=X; L=len(W)//2
    for i in range(L):
        z=h@W[2*i]+W[2*i+1]
        h=1.0/(1.0+np.exp(-np.clip(z,-30,30))) if i==L-1 else np.maximum(0,z); acts.append(h)
    return acts
def predict(W,X): return fwd_all(W,X)[-1].ravel()
def grads(W,Xb,yb):
    L=len(W)//2; acts=fwd_all(W,Xb); o=acts[-1].ravel()
    delta=((o-yb)/len(yb))[:,None]; g=[None]*(2*L)
    for i in range(L-1,-1,-1):
        g[2*i]=acts[i].T@delta; g[2*i+1]=delta.sum(0)
        if i>0: delta=(delta@W[2*i].T)*(acts[i]>0)
    return g
def local_train(W,X,y,epochs,rng,lr=None,mu=0.0,Wg=None):
    lr=LR if lr is None else lr
    W=[w.copy() for w in W]; m=[np.zeros_like(w) for w in W]; v=[np.zeros_like(w) for w in W]; t=0; n=len(y)
    for ep in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,BATCH):
            bi=idx[s:s+BATCH]; g=grads(W,X[bi],y[bi].astype(float))
            if mu>0: g=[gi+mu*(wi-wg) for gi,wi,wg in zip(g,W,Wg)]
            t+=1
            for i in range(len(W)):
                m[i]=0.9*m[i]+0.1*g[i]; v[i]=0.999*v[i]+0.001*g[i]**2
                W[i]-=lr*(m[i]/(1-0.9**t))/(np.sqrt(v[i]/(1-0.999**t))+1e-8)
    return W
def evaluate(W,X,y):
    p=predict(W,X); return float(matthews_corrcoef(y,(p>=0.5).astype(int))),float(roc_auc_score(y,p))

# ---------------- tampilan fitur per klien (normalisasi LOKAL: statistik train klien sendiri) ----------------
NORM={c: prep(DATA[c]['train'][0],DATA[c]['test'][0]) for c in CLIENTS}
def make_view(c,kind,A):
    n=len(A)
    if kind=='native': return A
    if kind=='int9': return A[:,:D_CORE]
    if kind=='pad':   # [inti 9 | ekstensi D_EXT (nol bila tak punya) | bendera punya-ekstensi]
        if c=='unsw': return np.hstack([A[:,:D_CORE],np.zeros((n,D_EXT)),np.zeros((n,1))])
        return np.hstack([A[:,:D_CORE],A[:,D_CORE:D_CORE+D_EXT],np.ones((n,1))])
V={k:{c:(make_view(c,k,NORM[c][0]),make_view(c,k,NORM[c][1])) for c in CLIENTS} for k in ('native','int9','pad')}
D_PAD=D_CORE+D_EXT+1; EXT_ROWS=slice(D_CORE,D_PAD)
YTR={c:DATA[c]['train'][1] for c in CLIENTS}; YTE={c:DATA[c]['test'][1] for c in CLIENTS}
print('dim: int9 =',V['int9']['cic2017'][0].shape[1],'| pad =',D_PAD,'| native cic/unsw =',V['native']['cic2017'][0].shape[1],V['native']['unsw'][0].shape[1])

def aggregate(Ws,ns,mode,owns):
    ns=np.asarray(ns,float); wa=ns/ns.sum()
    out=[sum(w*W[k] for w,W in zip(wa,Ws)) for k in range(len(Ws[0]))]
    if mode=='tier':   # baris ekstensi (+bendera) pada W1 dirata-rata hanya atas klien pemiliknya
        sel=[i for i,o in enumerate(owns) if o]; ws=ns[sel]/ns[sel].sum()
        out[0][EXT_ROWS]=sum(w*Ws[i][0][EXT_ROWS] for w,i in zip(ws,sel))
    return out

def run_fed(kind,mode,mu,seed):
    rng=np.random.RandomState(seed); d=V[kind][CLIENTS[0]][0].shape[1]; Wg=init_w([d]+HID+[1],seed)
    for r in range(ROUNDS):
        Ws=[];ns=[];owns=[]
        for c in CLIENTS:
            Ws.append(local_train(Wg,V[kind][c][0],YTR[c],LOCAL_EPOCHS,rng,mu=mu,Wg=Wg)); ns.append(len(YTR[c])); owns.append(c!='unsw')
        Wg=aggregate(Ws,ns,mode,owns)
    return {c:evaluate(Wg,V[kind][c][1],YTE[c]) for c in CLIENTS}
def run_fedper(seed):
    rng=np.random.RandomState(seed)
    Wc={c:init_w([V['native'][c][0].shape[1]]+HID+[1],seed) for c in CLIENTS}
    body=[w.copy() for w in Wc[CLIENTS[0]][2:]]
    for c in CLIENTS: Wc[c]=Wc[c][:2]+[w.copy() for w in body]
    for r in range(ROUNDS):
        tr={}
        for c in CLIENTS: tr[c]=local_train(Wc[c],V['native'][c][0],YTR[c],LOCAL_EPOCHS,rng)
        ns=np.array([len(YTR[c]) for c in CLIENTS],float); wa=ns/ns.sum()
        body=[sum(w*tr[c][k] for w,c in zip(wa,CLIENTS)) for k in range(2,len(tr[CLIENTS[0]]))]
        for c in CLIENTS: Wc[c]=tr[c][:2]+[w.copy() for w in body]
    return {c:evaluate(Wc[c],V['native'][c][1],YTE[c]) for c in CLIENTS}
def run_local(seed):
    rng=np.random.RandomState(seed); res={}
    for c in CLIENTS:
        X=V['native'][c][0]; W=local_train(init_w([X.shape[1]]+HID+[1],seed),X,YTR[c],LOCAL_ONLY_EPOCHS,rng)
        res[c]=evaluate(W,V['native'][c][1],YTE[c])
    return res
def run_central(seed):
    rng=np.random.RandomState(seed)
    X=np.vstack([V['pad'][c][0] for c in CLIENTS]); y=np.concatenate([YTR[c] for c in CLIENTS])
    W=local_train(init_w([D_PAD]+HID+[1],seed),X,y,CENTRAL_EPOCHS,rng)
    return {c:evaluate(W,V['pad'][c][1],YTE[c]) for c in CLIENTS}

M_LOCAL='M0 local-only'; M_CENT='M1 centralized-pad (acuan)'; M_INT9='M2 FedAvg-int9'; M_PAD='M3 FedAvg-pad76'
M_PER='M4 FedPer-ownlayer'; M_TIER='M5 FedAvg-FSFMtier'; M_PINT9='M6 FedProx-int9'; M_PPAD='M7 FedProx-pad76'; M_PTIER='M8 FedProx-FSFMtier'
METHODS={
 M_LOCAL: run_local, M_CENT: run_central,
 M_INT9:  lambda s: run_fed('int9','all',0.0,s),
 M_PAD:   lambda s: run_fed('pad','all',0.0,s),
 M_PER:   run_fedper,
 M_TIER:  lambda s: run_fed('pad','tier',0.0,s),
 M_PINT9: lambda s: run_fed('int9','all',MU,s),
 M_PPAD:  lambda s: run_fed('pad','all',MU,s),
 M_PTIER: lambda s: run_fed('pad','tier',MU,s),
}
print('mesin siap |',len(METHODS),'metode x',len(SEEDS),'seed')

In [ ]:
rows=[]; t00=time.time()
for seed in SEEDS:
    for name,fn in METHODS.items():
        t0=time.time(); res=fn(seed)
        for c,(m,a) in res.items(): rows.append(dict(method=name,seed=seed,client=c,mcc=round(m,4),auc=round(a,4)))
        print(f'seed {seed:3d} | {name:30s} | '+' '.join(f'{c}={res[c][0]:+.3f}' for c in CLIENTS)+f' | mean={np.mean([res[c][0] for c in CLIENTS]):+.3f} [{time.time()-t0:.0f}s]')
    pd.DataFrame(rows).to_csv(os.path.join(OUTDIR,'fsfm_compare_raw.csv'),index=False)   # simpan bertahap
df=pd.DataFrame(rows); df.to_csv(os.path.join(OUTDIR,'fsfm_compare_raw.csv'),index=False)
print('selesai dalam %.0f menit'%((time.time()-t00)/60))

In [ ]:
from scipy import stats
def ci(d):
    d=np.asarray(d,float); n=len(d); m=float(d.mean())
    if n<2: return m,float('nan'),float('nan')
    h=stats.t.ppf(0.975,n-1)*d.std(ddof=1)/np.sqrt(n); return m,m-h,m+h
cm=df.groupby(['method','seed']).mcc.mean().unstack(0)          # baris seed, kolom metode
print('=== MCC rata-rata tiga klien (mean, CI95) ===')
summ=[]
for name in METHODS:
    m,lo,hi=ci(cm[name]); summ.append(dict(method=name,mean_mcc=round(m,4),ci_lo=round(lo,4),ci_hi=round(hi,4)))
S=pd.DataFrame(summ); print(S.to_string(index=False))
print('\n=== MCC per klien (rata-rata seed) ===')
PC=df.pivot_table(index='method',columns='client',values='mcc',aggfunc='mean').round(3).loc[list(METHODS)]
print(PC.to_string())
print('\n=== AUC per klien (rata-rata seed) ===')
print(df.pivot_table(index='method',columns='client',values='auc',aggfunc='mean').round(3).loc[list(METHODS)].to_string())

def diff(a,b): return ci(cm[a]-cm[b])
d1=diff(M_TIER,M_INT9); du=ci(df[(df.client=='unsw')&(df.method==M_TIER)].set_index('seed').mcc-df[(df.client=='unsw')&(df.method==M_INT9)].set_index('seed').mcc)
H1=(d1[0]>=0.05 and d1[1]>0 and du[0]>=-0.02)
d2a=diff(M_TIER,M_PAD); d2b=diff(M_TIER,M_PER)
H2=(d2a[0]>=0.02 and d2a[1]>0 and d2b[0]>=0.02 and d2b[1]>0)
print('\n=== Kriteria (disepakati sebelum melihat angka) ===')
print(f'H1  M5-M2 = {d1[0]:+.3f} [{d1[1]:+.3f},{d1[2]:+.3f}] ; unsw M5-M2 = {du[0]:+.3f}  -> H1 {"LOLOS" if H1 else "GAGAL"}')
print(f'H2  M5-M3 = {d2a[0]:+.3f} [{d2a[1]:+.3f},{d2a[2]:+.3f}] ; M5-M4 = {d2b[0]:+.3f} [{d2b[1]:+.3f},{d2b[2]:+.3f}] -> H2 {"LOLOS" if H2 else "GAGAL"}')
extra=[('M8-M6 (FedProx tier vs int9)',diff(M_PTIER,M_PINT9)),('M8-M7 (FedProx tier vs pad)',diff(M_PTIER,M_PPAD)),('M5-M0 (tier vs local-only)',diff(M_TIER,M_LOCAL)),('M1-M5 (acuan vs tier)',diff(M_CENT,M_TIER))]
for n,(m,lo,hi) in extra: print(f'    {n:34s} = {m:+.3f} [{lo:+.3f},{hi:+.3f}]')
S.to_csv(os.path.join(OUTDIR,'fsfm_compare_summary.csv'),index=False); PC.to_csv(os.path.join(OUTDIR,'fsfm_compare_per_client.csv'))
json.dump(dict(H1=bool(H1),H2=bool(H2),d_M5_M2=d1,d_M5_M3=d2a,d_M5_M4=d2b,unsw_M5_M2=du,seeds=SEEDS,rounds=ROUNDS,mock=MOCK,fast=FAST),open(os.path.join(OUTDIR,'fsfm_criteria.json'),'w'),indent=1)
if not MOCK:
    for f in ('fsfm_compare_raw.csv','fsfm_compare_summary.csv','fsfm_compare_per_client.csv','fsfm_criteria.json'):
        s3.upload_file(os.path.join(OUTDIR,f),BUCKET,f'{OUT}/results/{f}')
    print('CSV/JSON tersimpan + upload ke s3://%s/%s/results/'%(BUCKET,OUT))
print('Jangan menyimpulkan sebelum Kiro membaca hasilnya.')

In [ ]:
if MOCK:
    print('MOCK: lewati cek S3')
else:
    NEED=[f'{OUT}/results/{f}' for f in ('fsfm_compare_raw.csv','fsfm_compare_summary.csv','fsfm_compare_per_client.csv','fsfm_criteria.json')]
    resp=s3.list_objects_v2(Bucket=BUCKET,Prefix=f'{OUT}/'); keys=sorted(o['Key'] for o in resp.get('Contents',[]))
    for k in keys: print('  s3://%s/%s'%(BUCKET,k))
    miss=[k for k in NEED if k not in keys]; print('LENGKAP' if not miss else 'KURANG: %s'%miss)
    print('Di komputer lokal (PowerShell): aws s3 sync s3://%s/%s/ ./fsfm'%(BUCKET,OUT))